# Exercice 5 — pandas : merge, valeurs manquantes, pivot_table

**Durée cible : 40-45 minutes**  
**Niveau : Junior**

---

## 💡 Tips

- `pd.merge(df1, df2, on="col", how="inner")` — fusionne deux DataFrames sur une colonne clé
  - `how="inner"` : garde seulement les lignes présentes dans les deux tables
  - `how="left"` : garde toutes les lignes de df1, NaN si pas de correspondance dans df2
  - `how="outer"` : garde toutes les lignes des deux tables
- `.isnull()` → masque booléen des valeurs manquantes / `.isnull().sum()` → compte par colonne
- `df.dropna()` → supprime les lignes avec au moins un NaN
- `df.fillna(valeur)` → remplace les NaN par une valeur (moyenne, 0, "Inconnu"...)
- `df["col"].fillna(df["col"].mean())` → remplace les NaN par la moyenne de la colonne
- `pd.pivot_table(df, values="col", index="row", columns="col2", aggfunc="sum")` — tableau croisé
- `.reset_index()` — transforme l'index en colonne ordinaire (utile après groupby/pivot)
- `df.rename(columns={"ancien": "nouveau"})` — renomme des colonnes

---


## Dataset

Exécute cette cellule pour créer les trois DataFrames de travail.

In [1]:
import pandas as pd
import numpy as np

# Table des commandes
commandes = pd.DataFrame({
    "commande_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "client_id":   [101, 102, 101, 103, 104, 102, 105, 103],
    "produit":     ["Laptop", "Chaise", "Ecran", "Clavier", "Laptop", "Bureau", "Ecran", "Lampe"],
    "quantite":    [1, 4, 2, 5, 1, 2, 3, 10],
    "prix_unitaire": [1200, 150, 400, 80, 1200, 600, 400, 45],
})

# Table des clients
clients = pd.DataFrame({
    "client_id": [101, 102, 103, 104, 106],
    "nom":       ["Alice", "Bob", "Clara", "David", "Eva"],
    "region":    ["Nord", "Sud", "Est", "Nord", "Ouest"],
    "segment":   ["Pro", "Particulier", "Pro", "Particulier", "Pro"],
})

# Table des livraisons (données incomplètes volontairement)
livraisons = pd.DataFrame({
    "commande_id":    [1, 2, 3, 4, 5, 6, 7, 8],
    "date_livraison": ["2024-01-10", None, "2024-01-15", "2024-01-12", None, "2024-01-20", None, "2024-01-18"],
    "transporteur":   ["DHL", "Chronopost", None, "DHL", None, "Chronopost", "DHL", None],
    "note_livraison": [4.5, None, 3.8, 5.0, None, 4.2, None, 3.5],
})

print("commandes :", commandes.shape)
print("clients   :", clients.shape)
print("livraisons:", livraisons.shape)


commandes : (8, 5)
clients   : (5, 4)
livraisons: (8, 4)


In [2]:
display(commandes)

,commande_id,client_id,produit,quantite,prix_unitaire
0,1,101,Laptop,1,1200
1,2,102,Chaise,4,150
2,3,101,Ecran,2,400
3,4,103,Clavier,5,80
4,5,104,Laptop,1,1200
5,6,102,Bureau,2,600
6,7,105,Ecran,3,400
7,8,103,Lampe,10,45


In [3]:
display(clients)

,client_id,nom,region,segment
0,101,Alice,Nord,Pro
1,102,Bob,Sud,Particulier
2,103,Clara,Est,Pro
3,104,David,Nord,Particulier
4,106,Eva,Ouest,Pro


---

## Partie 1 — Merge

### 1.1 — Inner join

Fusionne `commandes` et `clients` sur `client_id` avec un **inner join**.  
Stocke le résultat dans `df_inner`.

1. Combien de lignes contient `df_inner` ? Pourquoi pas 8 ?
2. Quel(s) `client_id` ont disparu ? Explique en une phrase pourquoi.

> 💡 Un inner join ne garde que les lignes qui ont une correspondance **dans les deux tables**.


In [4]:
print("Commandes columns:", commandes.columns.tolist())
print("Clients columns:", clients.columns.tolist())

Commandes columns: ['commande_id', 'client_id', 'produit', 'quantite', 'prix_unitaire']
Clients columns: ['client_id', 'nom', 'region', 'segment']


In [5]:
print("client_id in clients:", clients['client_id'])
print("client_id in commandes:", commandes['client_id'])

client_id in clients: 0    101
1    102
2    103
3    104
4    106
Name: client_id, dtype: int64
client_id in commandes: 0    101
1    102
2    101
3    103
4    104
5    102
6    105
7    103
Name: client_id, dtype: int64


In [6]:
df_inner = pd.merge(commandes, clients, how = 'inner', on = 'client_id')
display(df_inner)
print("Shape:", df_inner.shape)

,commande_id,client_id,produit,quantite,prix_unitaire,nom,region,segment
0,1,101,Laptop,1,1200,Alice,Nord,Pro
1,2,102,Chaise,4,150,Bob,Sud,Particulier
2,3,101,Ecran,2,400,Alice,Nord,Pro
3,4,103,Clavier,5,80,Clara,Est,Pro
4,5,104,Laptop,1,1200,David,Nord,Particulier
5,6,102,Bureau,2,600,Bob,Sud,Particulier
6,8,103,Lampe,10,45,Clara,Est,Pro


Shape: (7, 8)


*Ton interprétation ici (double-clique pour éditer) :*
105 est dans commandes mais absent de clients. L'inner join exige une correspondance dans les deux tables, donc la commande de 105 disparaît. 

### 1.2 — Left join

Fusionne `commandes` et `clients` avec un **left join**.  
Stocke le résultat dans `df_left`.

1. Combien de lignes contient `df_left` ?
2. Affiche les lignes où `nom` est NaN. Que représentent-elles ?


In [7]:
df_left = pd.merge(commandes, clients, how='left', on='client_id')
display(df_left)
print("number of rows:", df_left.shape[0])

,commande_id,client_id,produit,quantite,prix_unitaire,nom,region,segment
0,1,101,Laptop,1,1200,Alice,Nord,Pro
1,2,102,Chaise,4,150,Bob,Sud,Particulier
2,3,101,Ecran,2,400,Alice,Nord,Pro
3,4,103,Clavier,5,80,Clara,Est,Pro
4,5,104,Laptop,1,1200,David,Nord,Particulier
5,6,102,Bureau,2,600,Bob,Sud,Particulier
6,7,105,Ecran,3,400,NaN,NaN,NaN
7,8,103,Lampe,10,45,Clara,Est,Pro


number of rows: 8


In [8]:
print(df_left[df_left['nom'].isna()])

   commande_id  client_id produit  quantite  prix_unitaire  nom region segment
6            7        105   Ecran         3            400  NaN    NaN     NaN


*Ton interprétation ici :*
1. 8 lignes, car cette fois si on a mergé sur commandes. On a garder toutes les valeurs du dataframe commandes et on a rajouter celles du dataframe clients qui sont à l'intersection de commandes.
2. Ce sont des commandes passées par le client_id 105, qui n'a pas de fiche dans la table clients — ses infos (nom, région, segment) sont donc NaN.

### 1.3 — Triple merge

À partir de `df_left`, fusionne avec `livraisons` sur `commande_id` (left join).  
Stocke le résultat dans `df_full`.

Affiche `.shape` et `.columns` du résultat.


In [9]:
df_full = pd.merge(df_left, livraisons, how='left', on='commande_id')
display(df_full)
print("Shape:", df_full.shape)
print("Columns of df_full:", df_full.columns.tolist())

,commande_id,client_id,produit,quantite,prix_unitaire,nom,region,segment,date_livraison,transporteur,note_livraison
0,1,101,Laptop,1,1200,Alice,Nord,Pro,2024-01-10,DHL,4.5
1,2,102,Chaise,4,150,Bob,Sud,Particulier,None,Chronopost,NaN
2,3,101,Ecran,2,400,Alice,Nord,Pro,2024-01-15,None,3.8
3,4,103,Clavier,5,80,Clara,Est,Pro,2024-01-12,DHL,5.0
4,5,104,Laptop,1,1200,David,Nord,Particulier,None,None,NaN
5,6,102,Bureau,2,600,Bob,Sud,Particulier,2024-01-20,Chronopost,4.2
6,7,105,Ecran,3,400,NaN,NaN,NaN,None,DHL,NaN
7,8,103,Lampe,10,45,Clara,Est,Pro,2024-01-18,None,3.5


Shape: (8, 11)
Columns of df_full: ['commande_id', 'client_id', 'produit', 'quantite', 'prix_unitaire', 'nom', 'region', 'segment', 'date_livraison', 'transporteur', 'note_livraison']


---

## Partie 2 — Valeurs manquantes

Travaille sur `df_full` pour toute cette partie.

### 2.1 — Diagnostic

1. Affiche le **nombre de valeurs manquantes par colonne** (une seule ligne de code)
2. Affiche le **pourcentage** de valeurs manquantes par colonne (arrondi à 1 décimale)

> 💡 `isnull().sum()` donne le compte — comment obtenir le pourcentage ?


In [10]:
print("Number of missing values per columns:\n", df_full.isna().sum())

missing_percent_col = round((df_full.isna().sum() / df_full.shape[0]) * 100, 1)
print("\nPercentage of missing values per columns:\n", missing_percent_col)

Number of missing values per columns:
 commande_id       0
client_id         0
produit           0
quantite          0
prix_unitaire     0
nom               1
region            1
segment           1
date_livraison    3
transporteur      3
note_livraison    3
dtype: int64

Percentage of missing values per columns:
 commande_id        0.0
client_id          0.0
produit            0.0
quantite           0.0
prix_unitaire      0.0
nom               12.5
region            12.5
segment           12.5
date_livraison    37.5
transporteur      37.5
note_livraison    37.5
dtype: float64


### 2.2 — Traitement

1. Remplace les NaN de `note_livraison` par la **moyenne** de cette colonne
2. Remplace les NaN de `transporteur` par la chaîne `"Inconnu"`
3. Remplace les NaN de `date_livraison` par la chaîne `"Non livré"`
4. Remplace les NaN de `nom` par `"Client inconnu"`
5. Vérifie qu'il ne reste plus de NaN dans ces 4 colonnes

> ⚠️ N'utilise pas `df_full = df_full.fillna(...)` globalement — traite **chaque colonne séparément** avec la stratégie adaptée.


In [11]:
type(df_full)

pandas.core.frame.DataFrame

In [12]:
mean_note_livraison = df_full['note_livraison'].mean()
df_full['note_livraison'] = np.where(df_full['note_livraison'].isna(), mean_note_livraison, df_full['note_livraison'])
df_full['transporteur'] = np.where(df_full['transporteur'].isna(), 'Inconnu', df_full['transporteur'])
df_full['date_livraison'] = np.where(df_full['date_livraison'].isna(), 'Non livré', df_full['date_livraison'])
df_full['nom'] = np.where(df_full['nom'].isna(), 'Client inconnu', df_full['nom'])

In [13]:
display(df_full)

,commande_id,client_id,produit,quantite,prix_unitaire,nom,region,segment,date_livraison,transporteur,note_livraison
0,1,101,Laptop,1,1200,Alice,Nord,Pro,2024-01-10,DHL,4.5
1,2,102,Chaise,4,150,Bob,Sud,Particulier,Non livré,Chronopost,4.2
2,3,101,Ecran,2,400,Alice,Nord,Pro,2024-01-15,Inconnu,3.8
3,4,103,Clavier,5,80,Clara,Est,Pro,2024-01-12,DHL,5.0
4,5,104,Laptop,1,1200,David,Nord,Particulier,Non livré,Inconnu,4.2
5,6,102,Bureau,2,600,Bob,Sud,Particulier,2024-01-20,Chronopost,4.2
6,7,105,Ecran,3,400,Client inconnu,NaN,NaN,Non livré,DHL,4.2
7,8,103,Lampe,10,45,Clara,Est,Pro,2024-01-18,Inconnu,3.5


In [14]:
mean_note_livraison

np.float64(4.2)

---

## Partie 3 — Colonnes calculées & analyse

### 3.1 — Chiffre d'affaires

1. Crée une colonne `chiffre_affaires` = `quantite * prix_unitaire` dans `df_full`
2. Affiche le CA total de tout le dataset
3. Affiche le CA total **par région** (groupby + sum) — trie par CA décroissant

> 💡 Les clients avec `nom = "Client inconnu"` ont une région NaN — que se passe-t-il dans le groupby ? C'est normal.


In [15]:
df_full['chiffre_affaires'] = df_full['quantite'] * df_full['prix_unitaire']
ca_total = df_full['chiffre_affaires'].sum()
print("Chiffre d'affaire total:", ca_total)

ca_total_region = df_full.groupby('region')['chiffre_affaires'].sum().sort_values(ascending=False)
print("\nCA total per region:", ca_total_region)

Chiffre d'affaire total: 7050

CA total per region: region
Nord    3200
Sud     1800
Est      850
Name: chiffre_affaires, dtype: int64


*Ton interprétation ici :*
Le CA total est 7 050 €, le Nord domine (3 200 €), suivi du Sud (1 800 €) et de l'Est (850 €). Le client_id 105 (région NaN) est exclu du groupby automatiquement.

### 3.2 — Pivot table

Crée un **tableau croisé** qui affiche le **CA total** par `region` (en lignes) et par `transporteur` (en colonnes).

- Utilise `pd.pivot_table()`
- Les valeurs manquantes dans le pivot peuvent rester NaN (c'est informatif)
- Affiche le résultat

Ensuite, réponds : quelle région + transporteur génère le CA le plus élevé ?


In [16]:
pivot = pd.pivot_table(data = df_full,
                       values='chiffre_affaires',
                       index='region',
                       columns='transporteur',
                       aggfunc='sum')
display(pivot)

transporteur,Chronopost,DHL,Inconnu
region,,,
Est,NaN,400.0,450.0
Nord,NaN,1200.0,2000.0
Sud,1800.0,NaN,NaN


*Ton interprétation ici :*
Nord + Inconnu = 2 000 € est supérieur

---

## 🎯 Bonus

1. Affiche les commandes dont le `transporteur` était `"Inconnu"` (donc NaN à l'origine) **ET** dont le CA est supérieur à 500 €
2. Calcule, par `segment` client (`Pro` / `Particulier`), la **note de livraison moyenne** (après remplacement des NaN) — trie par note décroissante
3. Renomme les colonnes `prix_unitaire` → `prix` et `chiffre_affaires` → `ca` dans `df_full` puis affiche `.columns`

> 💡 Bonus 1 : filtre sur la valeur `"Inconnu"` (tu l'as déjà remplacé), pas sur NaN


In [17]:
# Ton code ici
mask = df_full[(df_full["transporteur"] == "Inconnu") & (df_full["chiffre_affaires"] > 500)]
print(mask)

   commande_id  client_id produit  quantite  prix_unitaire    nom region  \
2            3        101   Ecran         2            400  Alice   Nord   
4            5        104  Laptop         1           1200  David   Nord   

       segment date_livraison transporteur  note_livraison  chiffre_affaires  
2          Pro     2024-01-15      Inconnu             3.8               800  
4  Particulier      Non livré      Inconnu             4.2              1200  


In [18]:
print(df_full.groupby("segment")["note_livraison"].mean().sort_values(ascending=False))

segment
Particulier    4.2
Pro            4.2
Name: note_livraison, dtype: float64


In [19]:
df_full = df_full.rename(columns={"prix_unitaire": "prix", "chiffre_affaires": "ca"})
print(df_full.columns.tolist())

['commande_id', 'client_id', 'produit', 'quantite', 'prix', 'nom', 'region', 'segment', 'date_livraison', 'transporteur', 'note_livraison', 'ca']
